# Govern Unity Catalog objects

> **Industry scenario — AutoNova Group**
>
> AutoNova Group is a global automotive manufacturer selling connected vehicles across four regions: North America, Europe, Asia Pacific, and South America. Their Unity Catalog environment (catalog `trainer_demo`, schema `demo_05`) holds four tables:
> - **`vehicles`** — vehicle inventory with VINs, make, model, and year
> - **`customers`** — registered owners including PII (email, phone)
> - **`telemetry_events`** — real-time sensor data streamed from connected vehicles
> - **`service_records`** — dealership service history
>
> As a data governance engineer you will document table definitions, apply access policies, manage data retention, trace data lineage, audit access, and design a secure Delta Sharing strategy with dealer partners.

In [ ]:
# Run the setup script to create sample automotive data
from scripts.setup import setup
setup(spark)

## Create and preserve table definitions

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/govern-unity-catalog-objects.preserve-table-column-definitions.png)

### Demo: Add comments and tags to AutoNova tables

**Instructor note:** Run the following cells in sequence. These cells demonstrate:
1. Viewing the existing automotive tables
2. Adding descriptive comments to tables and columns
3. Applying tags for classification and discovery
4. Using `DESCRIBE TABLE EXTENDED` to verify metadata

In [ ]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_05;

-- Explore the available tables
SHOW TABLES;

In [ ]:
%sql
-- Inspect the vehicles table schema
DESCRIBE TABLE EXTENDED vehicles;

In [ ]:
%sql
-- Add a table-level comment describing the vehicles table
COMMENT ON TABLE vehicles
IS '## AutoNova Vehicle Inventory
Contains all registered vehicles manufactured by AutoNova Group.
- **vin**: Vehicle Identification Number – globally unique per vehicle
- **vehicle_type**: Sedan, SUV, Truck, Electric, Hatchback, Coupe
- **region**: Sales region where vehicle is registered';

-- Add column-level comments
ALTER TABLE vehicles ALTER COLUMN vin     COMMENT 'Vehicle Identification Number — 17-character unique identifier';
ALTER TABLE vehicles ALTER COLUMN vehicle_type COMMENT 'Body style classification: Sedan, SUV, Truck, Electric, Hatchback, Coupe';
ALTER TABLE vehicles ALTER COLUMN region  COMMENT 'Primary sales region: North America, Europe, Asia Pacific, South America';

-- Add comments to the customers table (contains PII)
COMMENT ON TABLE customers
IS '## AutoNova Customer Registry
Registered vehicle owners. Contains PII — handle in accordance with GDPR and regional data protection laws.
- **email**: Primary contact email address
- **phone**: International format phone number';

ALTER TABLE customers ALTER COLUMN email  COMMENT 'Primary email address — PII, subject to right-to-erasure requests';
ALTER TABLE customers ALTER COLUMN phone  COMMENT 'International phone number — PII, subject to right-to-erasure requests';

-- Add comments to telemetry_events
COMMENT ON TABLE telemetry_events
IS '## Connected Vehicle Telemetry
Real-time sensor data emitted by AutoNova connected vehicles every 5 minutes.
- **event_type**: normal | warning | critical | idle
- **engine_temp_c**: Engine temperature in Celsius
- **battery_level_pct**: Battery state of charge (0–100)';

In [ ]:
%sql
-- Add classification tags to the vehicles table
ALTER TABLE vehicles
SET TAGS ('domain' = 'vehicle_operations', 'data_classification' = 'internal');

-- Tag the customers table — it contains PII
ALTER TABLE customers
SET TAGS ('domain' = 'customer_data', 'data_classification' = 'confidential');

-- Tag individual PII columns
ALTER TABLE customers ALTER COLUMN email
SET TAGS ('pii' = 'email', 'sensitivity' = 'high');

ALTER TABLE customers ALTER COLUMN phone
SET TAGS ('pii' = 'phone', 'sensitivity' = 'high');

-- Tag the VIN as a quasi-identifier
ALTER TABLE vehicles ALTER COLUMN vin
SET TAGS ('sensitivity' = 'medium', 'identifier' = 'vehicle_vin');

-- Tag telemetry for operational use
ALTER TABLE telemetry_events
SET TAGS ('domain' = 'telematics', 'data_classification' = 'internal');

-- Verify tags on the customers table
DESCRIBE TABLE EXTENDED customers;

In [ ]:
%sql
-- Use SQL to explore documented assets
SHOW CATALOGS;
SHOW SCHEMAS IN trainer_demo;

-- View tagged columns in the customers table
SHOW COLUMNS IN customers;

-- Check the service_records table definition — add a comment
COMMENT ON TABLE service_records
IS '## Vehicle Service History
Dealership service and repair records for AutoNova vehicles. Used for warranty analytics and predictive maintenance.
- **service_type**: Type of service performed (Oil Change, Battery Check, etc.)
- **cost_eur**: Service cost in Euros
- **dealer_id**: Authorised AutoNova dealership identifier';

## Configure ABAC with tags and policies

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/govern-unity-catalog-objects.configure-abac-tags-policies.png)

### Demo: Attribute-based access control with governed tags

**Instructor note:** This section demonstrates ABAC using governed tags. In a real environment:
1. An admin creates **governed tags** in Account Console → Catalog → Governance → Governed Tags
2. Those tags are applied to sensitive columns
3. Column mask and row filter **policies** are created via SQL or Catalog Explorer

The SQL below shows the concepts. Policy creation (`CREATE POLICY`) requires Public Preview features to be enabled on the metastore.

In [ ]:
%sql
-- STEP 1: Apply governed tags to PII columns
-- (Assume governed tag 'pii' with allowed values 'email', 'phone', 'vin' was
--  created in Account Console by an admin)

ALTER TABLE customers ALTER COLUMN email
SET TAGS ('pii' = 'email');

ALTER TABLE customers ALTER COLUMN phone
SET TAGS ('pii' = 'phone');

ALTER TABLE vehicles ALTER COLUMN vin
SET TAGS ('pii' = 'vin');

In [ ]:
%sql
-- STEP 2: Create a masking function for email addresses
-- Returns masked email for users who are NOT in the 'data_stewards' group

CREATE OR REPLACE FUNCTION trainer_demo.demo_05.mask_email(email STRING)
RETURNS STRING
DETERMINISTIC
RETURN CASE
    WHEN is_account_group_member('data_stewards') THEN email
    ELSE CONCAT(LEFT(email, 2), '***@***.com')
END;

-- Test the function (when NOT in data_stewards, you see masked output)
SELECT trainer_demo.demo_05.mask_email('luca.rossi99@autonova-demo.com') AS masked_email;

In [ ]:
%sql
-- STEP 3: Create a column mask policy using the UDF and governed tag
-- This policy automatically applies to any column tagged pii=email in the schema
-- NOTE: Requires ABAC Public Preview to be enabled on your metastore

CREATE POLICY IF NOT EXISTS mask_customer_email
ON SCHEMA trainer_demo.demo_05
COLUMN MASK trainer_demo.demo_05.mask_email
TO `all_account_users`
EXCEPT `data_stewards`
FOR TABLES
MATCH COLUMNS
    hasTagValue('pii', 'email') AS email_col
ON COLUMN email_col;

In [ ]:
%sql
-- STEP 4: Row filter — regional compliance example
-- Only analysts in the 'eu_analysts' group should see European customer records
-- Others see only non-European customers

CREATE OR REPLACE FUNCTION trainer_demo.demo_05.filter_eu_region(region STRING)
RETURNS BOOLEAN
RETURN CASE
    WHEN is_account_group_member('eu_analysts') THEN TRUE
    ELSE region != 'Europe'
END;

-- Apply as a row filter policy on the customers table
ALTER TABLE customers SET ROW FILTER trainer_demo.demo_05.filter_eu_region ON (region);

-- Verify: non-EU analysts should not see European rows
SELECT region, COUNT(*) AS customer_count
FROM customers
GROUP BY region
ORDER BY region;

## Apply data retention policies

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/govern-unity-catalog-objects/media/5-understand-data-retention-settings.png)

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/govern-unity-catalog-objects.apply-data-retention-policies.png)

### Demo: Data retention policies for vehicle telemetry

**Instructor note:** AutoNova retains telemetry data for 90 days for operational use, but GDPR requires the ability to permanently delete personal data (customers) on request. This section demonstrates:
1. Viewing and setting Delta Lake retention properties
2. Using `VACUUM` to purge obsolete data files
3. Processing GDPR right-to-erasure requests
4. Enabling predictive optimization for automated maintenance

In [ ]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_05;

-- View current retention settings on the telemetry table
SHOW TBLPROPERTIES telemetry_events;

In [ ]:
%sql
-- Set retention on telemetry_events to 90 days
-- AutoNova only needs 90 days of telemetry history for SLA reporting
ALTER TABLE telemetry_events
SET TBLPROPERTIES (
    'delta.logRetentionDuration'        = 'interval 90 days',
    'delta.deletedFileRetentionDuration' = 'interval 90 days'
);

-- Set longer retention on service_records for warranty compliance (7 years)
ALTER TABLE service_records
SET TBLPROPERTIES (
    'delta.logRetentionDuration'        = 'interval 2555 days',
    'delta.deletedFileRetentionDuration' = 'interval 2555 days'
);

SHOW TBLPROPERTIES telemetry_events;

In [ ]:
%sql
-- VACUUM: remove data files older than the retention window
-- (DRY RUN first — shows what would be deleted without actually deleting)
VACUUM telemetry_events DRY RUN;

-- Run actual VACUUM after confirming the dry run output
-- VACUUM telemetry_events;

In [ ]:
%sql
-- GDPR right-to-erasure: delete a customer and their telemetry
-- Simulates a deletion request for customer_id = 42

-- Step 1: Delete the customer record
DELETE FROM customers WHERE customer_id = 42;

-- Step 2: Delete their telemetry (linked via vehicle_id)
DELETE FROM telemetry_events
WHERE vehicle_id IN (
    SELECT vehicle_id FROM customers WHERE customer_id = 42
);

-- Step 3: Verify deletion
SELECT COUNT(*) AS remaining_records FROM customers WHERE customer_id = 42;

-- Step 4: Run VACUUM to permanently remove data files (irreversible!)
-- VACUUM customers;
-- VACUUM telemetry_events;

In [ ]:
%sql
-- Enable predictive optimization on demo_05 schema
-- This automates VACUUM and OPTIMIZE across all managed tables
ALTER SCHEMA trainer_demo.demo_05
ENABLE PREDICTIVE OPTIMIZATION;

-- Check if predictive optimization is active on the telemetry table
DESCRIBE TABLE EXTENDED telemetry_events;

## Set up and manage data lineage

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/govern-unity-catalog-objects.manage-data-lineage-tracking.png)

### Demo: Data lineage in Unity Catalog

**Instructor note:** Unity Catalog automatically captures lineage for queries run on Unity Catalog-enabled compute. This section demonstrates:
1. Creating a derived table to generate lineage relationships
2. Viewing table history with `DESCRIBE HISTORY`
3. Querying system lineage tables (`system.access.table_lineage`)

**Tip:** Show the **Lineage** tab in Catalog Explorer on `vehicles_with_service` while running the cells.

In [ ]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_05;

-- Create a summary view that derives from vehicles, customers, and service_records
-- This will appear as a downstream node in the lineage graph
CREATE OR REPLACE TABLE vehicles_with_service AS
SELECT
    v.vehicle_id,
    v.vin,
    v.make,
    v.model,
    v.year,
    v.vehicle_type,
    c.region,
    COUNT(s.service_id)     AS total_services,
    SUM(s.cost_eur)         AS total_service_cost_eur,
    MAX(s.service_date)     AS last_service_date
FROM vehicles v
LEFT JOIN service_records s ON v.vehicle_id = s.vehicle_id
LEFT JOIN customers c       ON s.customer_id = c.customer_id
GROUP BY v.vehicle_id, v.vin, v.make, v.model, v.year, v.vehicle_type, c.region;

SELECT * FROM vehicles_with_service LIMIT 5;

In [ ]:
%sql
-- View the change history of the vehicles table
DESCRIBE HISTORY vehicles;

-- Show only the latest 5 versions
DESCRIBE HISTORY vehicles LIMIT 5;

In [ ]:
%sql
-- Query Unity Catalog system lineage tables
-- Shows which tables were read/written in the last 7 days

SELECT
    source_table_full_name,
    target_table_full_name,
    COUNT(DISTINCT event_id)  AS event_count,
    MAX(event_time)           AS last_event_time
FROM system.access.table_lineage
WHERE event_date > CURRENT_DATE() - INTERVAL 7 DAYS
  AND (
        source_table_full_name LIKE 'trainer_demo.demo_05.%'
     OR target_table_full_name LIKE 'trainer_demo.demo_05.%'
  )
GROUP BY source_table_full_name, target_table_full_name
ORDER BY last_event_time DESC;

In [ ]:
%sql
-- View and change table ownership
-- (Requires metastore admin or current owner role)
DESCRIBE TABLE EXTENDED vehicles_with_service;

-- Transfer ownership example (replace with actual user/group)
-- ALTER TABLE vehicles_with_service OWNER TO `data-engineering-team@autonova.com`;

## Configure audit logging

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/govern-unity-catalog-objects.configure-audit-logging.png)

### Demo: Audit logging with system.access.audit

**Instructor note:** Azure Databricks stores all audit events in `system.access.audit`. This section demonstrates common governance queries AutoNova's compliance team would run:
1. Find who accessed sensitive customer data
2. Track permission changes across the metastore
3. Identify suspicious failed access attempts
4. Monitor notebook command execution (verbose logging)

In [ ]:
%sql
-- Find who queried the customers table (contains PII) in the last 7 days
SELECT
    user_identity.email                     AS user_email,
    action_name                             AS access_type,
    event_time                              AS access_time,
    source_ip_address
FROM system.access.audit
WHERE
    request_params.full_name_arg = 'trainer_demo.demo_05.customers'
    AND action_name IN ('getTable', 'createTable', 'deleteTable', 'updateTable')
    AND event_date > CURRENT_DATE() - INTERVAL 7 DAYS
ORDER BY event_time DESC
LIMIT 50;

In [ ]:
%sql
-- Track all permission changes across the trainer_demo catalog
-- Useful for auditing who granted/revoked access to vehicle and customer data
SELECT
    event_time,
    user_identity.email                     AS changed_by,
    request_params.securable_type           AS object_type,
    request_params.securable_full_name      AS object_name,
    request_params.changes                  AS permission_changes
FROM system.access.audit
WHERE
    service_name  = 'unityCatalog'
    AND action_name = 'updatePermissions'
    AND request_params.securable_full_name LIKE 'trainer_demo%'
ORDER BY event_time DESC
LIMIT 30;

In [ ]:
%sql
-- Identify failed access attempts — potential security incident indicator
SELECT
    event_time,
    user_identity.email                     AS user_email,
    source_ip_address,
    action_name,
    response.error_message
FROM system.access.audit
WHERE
    service_name          = 'accounts'
    AND response.status_code != 200
    AND event_date > CURRENT_DATE() - INTERVAL 30 DAYS
ORDER BY event_time DESC
LIMIT 30;

In [ ]:
%sql
-- Monitor job and cluster activity related to AutoNova data pipelines
SELECT
    event_time,
    user_identity.email                     AS triggered_by,
    action_name,
    request_params.job_id                   AS job_id,
    response.status_code
FROM system.access.audit
WHERE
    service_name = 'jobs'
    AND event_date > CURRENT_DATE() - INTERVAL 7 DAYS
ORDER BY event_time DESC
LIMIT 30;

## Design secure Delta Sharing strategy

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/govern-unity-catalog-objects.secure-delta-sharing-strategy.png)

### Demo: Secure Delta Sharing with dealer partners

**Instructor note:** AutoNova shares aggregated vehicle service data with authorised dealerships via Delta Sharing. Dealers need access to service summaries for their vehicles — but NOT raw customer PII. This section demonstrates:
1. Creating a share and adding an aggregated view
2. Creating a Databricks-to-Databricks recipient
3. Granting the recipient access to the share
4. Monitoring share activity

In [ ]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_05;

-- Create a dealer-safe view: no PII, only vehicle service summaries
CREATE OR REPLACE VIEW dealer_service_summary AS
SELECT
    s.dealer_id,
    v.make,
    v.model,
    v.year,
    v.vehicle_type,
    s.service_type,
    AVG(s.cost_eur)          AS avg_cost_eur,
    COUNT(s.service_id)      AS service_count,
    MAX(s.service_date)      AS latest_service_date
FROM service_records s
JOIN vehicles v ON s.vehicle_id = v.vehicle_id
GROUP BY s.dealer_id, v.make, v.model, v.year, v.vehicle_type, s.service_type;

SELECT * FROM dealer_service_summary LIMIT 10;

In [ ]:
%sql
-- Create a Delta Share for authorised dealer partners
CREATE SHARE IF NOT EXISTS autonova_dealer_share
COMMENT 'Aggregated vehicle service data for authorised AutoNova dealerships — no PII included';

-- Add the dealer-safe view to the share
ALTER SHARE autonova_dealer_share
ADD VIEW trainer_demo.demo_05.dealer_service_summary
COMMENT 'Service cost and frequency summary by dealer, vehicle type, and service type';

-- Optionally share the vehicles table (no PII columns)
ALTER SHARE autonova_dealer_share
ADD TABLE trainer_demo.demo_05.vehicles;

-- Inspect what is in the share
SHOW ALL IN SHARE autonova_dealer_share;

In [ ]:
%sql
-- Create a Databricks-to-Databricks recipient for a dealer network partner
-- The recipient provides their metastore sharing identifier:
--   SELECT CURRENT_METASTORE();  (run in their workspace)

CREATE RECIPIENT IF NOT EXISTS contoso_dealer_network
USING ID 'azure:westeurope:00000000-0000-0000-0000-000000000000'  -- replace with actual ID
COMMENT 'Contoso Dealer Network — authorised AutoNova partner';

-- Grant the recipient access to the share
GRANT SELECT ON SHARE autonova_dealer_share TO RECIPIENT contoso_dealer_network;

-- Verify grants
SHOW GRANT ON SHARE autonova_dealer_share;

In [ ]:
%sql
-- Monitor share access: who accessed the share and when
SELECT
    event_time,
    user_identity.email                     AS recipient_user,
    action_name,
    request_params.share_name
FROM system.access.audit
WHERE
    service_name  = 'deltaSharingServer'
    AND event_date > CURRENT_DATE() - INTERVAL 30 DAYS
ORDER BY event_time DESC
LIMIT 30;

In [ ]:
%sql
-- Clean-up / management commands (reference only — do not run in demo)

-- Remove a table from a share
-- ALTER SHARE autonova_dealer_share REMOVE TABLE trainer_demo.demo_05.vehicles;

-- Revoke a recipient's access
-- REVOKE SELECT ON SHARE autonova_dealer_share FROM RECIPIENT contoso_dealer_network;

-- Drop a share (removes all assets and access grants)
-- DROP SHARE IF EXISTS autonova_dealer_share;

-- List all shares visible to you
SHOW SHARES;